In [1]:
# Librerias
import pandas as pd
import numpy as np

In [ ]:
# Carga de los datos

RUTA_CR = "../datos/CR.txt"

COLUMNAS = [
    "geonameid", "nombre", "nombre_ascii", "nombres_alternos", "latitud", "longitud",
    "clase", "codigo", "pais", "cc2", "admin1", "admin2", "admin3", "admin4",
    "poblacion", "elevacion", "dem", "zona_horaria", "fecha_modificacion",
]

def leer_geonames(ruta=RUTA_CR):
    df = pd.read_csv(
        ruta,
        sep="\t",
        header=None,
        names=COLUMNAS,
        dtype={"admin1": str, "admin2": str, "admin3": str, "admin4": str, "cc2": str},
    )
    return df

df = leer_geonames()

In [ ]:
# filtrado

pob_min = 5000  # Mínimo de población (hay muchos en cero)

# solo poblados (city, village, ...)
df_filtrado = df[df["clase"] == "P"]

# excluir poblados destruidos o históricos
df_filtrado = df_filtrado[~df_filtrado["codigo"].isin(["PPLQ", "PPLW", "PPLH"])]

# umbral de población
df_filtrado = df_filtrado[df_filtrado["poblacion"] >= pob_min]

# quitar coordenadas duplicadas
df_filtrado = df_filtrado.drop_duplicates(subset=["latitud", "longitud"])

# ordenar por población y renumerar el índice
df_filtrado = df_filtrado.sort_values("poblacion", ascending=False).reset_index(drop=True)

df_filtrado.head()

,geonameid,name,asciiname,alternatenames,latitude,longitude,class,code,country,cc2,admin1,admin2,admin3,admin4,population,elevation,dem,timezone,modification_date
0,3621849,San José,San Jose,"SJO,San Chose,San Chosė,San Hose,San Jose,San ...",9.93388,-84.08489,P,PPLC,CR,NaN,08,101,10103,NaN,335007,NaN,1133,America/Costa_Rica,2025-05-15
1,3622247,Limón,Limon,"Ciudad de Limon,Ciudad de Limón,LIO,Limon,Limo...",9.99074,-83.03596,P,PPLA,CR,NaN,06,701,70101,NaN,63081,NaN,29,America/Costa_Rica,2026-04-13
2,3621911,San Francisco,San Francisco,San Francisco,9.99380,-84.12742,P,PPLX,CR,NaN,04,401,40103,NaN,55923,NaN,1131,America/Costa_Rica,2021-11-11
3,3624955,Alajuela,Alajuela,"Alachouela,Alachuela,Alajuela,Alakhuehla,Alaxu...",10.01723,-84.21275,P,PPLA,CR,NaN,01,201,20101,NaN,47494,NaN,955,America/Costa_Rica,2026-04-13
4,3623076,Liberia,Liberia,"LIR,Liberia,Liberii,Liberija,Liberiya,Liberya,...",10.63517,-85.43779,P,PPLA,CR,NaN,03,501,50101,NaN,45380,NaN,148,America/Costa_Rica,2026-04-13


# Y-opt

El algoritmo Y-opt es una variante del algoritmo 2-opt. Como se vio anteriormente, el algoritmo 2-opt acepta un intercambio de aristas si la suma de las longitudes de las dos aristas nuevas es menor que la suma de las dos aristas viejas; esto se hace comparando únicamente las aristas que se modificaron, ya que el resto de la ruta permanece igual. Por otro lado, el algoritmo Y-opt impone una condición de cambio más estricta, donde, para aceptar un cambio en la ruta, la arista más larga de las nuevas debe ser más corta que la arista más larga de las viejas, y, además, la arista más corta de las nuevas debe ser más corta que la arista más corta de las viejas. Esto hace que el algoritmo Y-opt sea más restrictivo en su condición de aceptación de nuevas rutas, ya que no solo compara la suma total, sino el tamaño individual de cada arista.

La principal ventaja del algoritmo Y-opt es que el número de iteraciones para una ruta de $n$ vértices y $m$ aristas está acotado por $(m-1)n$ (Manthey & van Rhijn, 2025), resultando en un tiempo de ejecución polinomial. En comparación, el algoritmo 2-opt puede necesitar un tiempo exponencial en los peores casos; de esta forma, el algoritmo Y-opt compensa su condición de aceptación con el tiempo que necesitará para ejecutarse. A pesar de que el algoritmo Y-opt sea más estricto, los resultados teóricos muestran que este algoritmo tiene resultados similares a los del 2-opt, donde se llega a rutas ligeramente más largas con el algoritmo Y-opt pero en un tiempo de ejecución acotado en comparación al 2-opt (Manthey & van Rhijn, 2025).

In [ ]:
# Funcion Y-opt

def y_opt(ruta_inicial, matriz_distancias):
    """
    Función para implementar el algoritmo Y-opt
    Entradas :           ruta_inicial ---- Lista con los indices de las ciudades 
                    matriz_distancias ---- Matriz de distancias de tamanyo nxn
                    
    Salidas :             ruta_nueva ---- Lista con la nueva ruta optimizada
    """
    
    n = len(ruta_inicial)
    ruta_copia = ruta_inicial.copy() # Copy para evitar sobreescribir la lista original
    mejorada = True 
    
    while mejorada == True: # Hacer el while hasta que la ruta no se pueda mejorar mas
        mejorada = False # Cambiar la condicion de si se mejoro o no la ruta (Se va a cambiar más adelante)
        
        for i in range(n-1):
            for j in range(i + 2, n): # i + 2 para evitar usar la misma arista en las comparaciones
                if (j == n-1): # Se llega a la última ciudad, la cual debe conectar con la primera siempre
                    k = 0
                else:
                    k = j + 1 # Si no estamos en la última ciudad, la ciudad actual se conecta con la siguiente
               
                # Las 4 ciudades consecutivas son: (Ordenadas segun la ruta inicial)
                a = ruta_copia[i]
                b = ruta_copia[i+1]
                c = ruta_copia[j]
                d = ruta_copia[k]
                
                # Largo de las aristas viejas:
                dist_vieja1 = matriz_distancias[a,b]
                dist_vieja2 = matriz_distancias[c,d]
                
                # Largo de las aristas nuevas:
                dist_nueva1 = matriz_distancias[a,c]
                dist_nueva2 = matriz_distancias[b,d]
                
                # Crear lista con las distancias nuevas
                
                dist_viejas = np.array([dist_vieja1,dist_vieja2])
                dist_nuevas = np.array([dist_nueva1,dist_nueva2])
                
                if (np.max(dist_nuevas) < np.max(dist_viejas)) and (np.min(dist_nuevas) < np.min(dist_viejas)): # Si esto se cumple se puede cambiar la ruta
                    ruta_nueva = ruta_copia[:i+1] + ruta_copia[i+1:j+1][::-1] + ruta_copia[j+1:] # Los + son para concatenar las listas, hay que recordar que en Python las listas empiezan en 0 y se incluye el extremo izquierdo pero no el derecho cuando se extrae una sublista; el [::-1] es para darle vuelta al segmento que está entre las aristas cambiadas
                    ruta_copia = ruta_nueva
                    mejorada = True
                    break # Salir del for de j
            if mejorada == True:
                break  # Salir del for de i
          
    return ruta_copia
            
            
    